# Arquitectura y modelo - DataByteV02.7 Chat-bot

## Flujo de datos

1. **Ingesta** — El comerciante carga lotes (foto de EAN / factura PDF) y ventas (archivo o foto de cuaderno).
2. **Automatización** — El bot llama a webhooks de **n8n**, que consultan el catálogo de la competencia y escriben en **Google Sheets**.
3. **Almacenamiento** — Pestañas: `Usuarios_Registrados`, `Lotes_Registrados`, `Ventas_Registradas`, `Score_Rescate`.
4. **Modelo** — `modelo_score.py` calcula el ranking y la acción recomendada.
5. **Visualización** — `exportar_dashboard.py` publica `Score_Rescate` → Looker Studio.

## Reglas de integridad de datos

- Cada comerciante solo ve y modifica **sus** lotes (aislamiento por `telegram_id`).
- Python solo escribe `cantidad`, `fecha_actualizacion` y `tiempo_restante` en `Lotes_Registrados`.
- Toda carga de ventas pasa por **vista previa + confirmación**.
- `registro_local.py` avisa si se sube dos veces el mismo archivo (evita descontar doble).
- Lo leído del cuaderno por IA nunca se guarda sin confirmación humana.

## Score de Rescate (0–100)

| Componente | Peso | Cálculo |
|---|---|---|
| Urgencia | 40 % | 100 si vence hoy/vencido; baja linealmente hasta 0 en `2 × DIAS_NARANJA` días |
| Ritmo | 25 % | % del lote que no se alcanzaría a vender (media móvil 7 días, proyectada al vencimiento) |
| Valor | 20 % | `cantidad × precio`, escalado hasta `VALOR_LOTE_ALTO` |
| Precio | 15 % | Distancia entre el precio actual y el descuento recomendado sobre el precio de la competencia |

Si falta un componente (sin historial o sin catálogo) se excluye y se **renormalizan los pesos**; la columna `datos_usados` indica la confianza.

**Niveles:** 🔴 ≥ 70 · 🟠 ≥ 40 · 🟢 menos.

**Acciones (por prioridad):** `RETIRAR` (vencido) → `LIQUIDAR_DONAR` (vence ≤ 1 día y merma ≥ 50 %) → `BAJAR_YA` / `BAJAR` → `REFORZAR` (el precio ya es competitivo) → `MANTENER`.

**Descuento sugerido:** `50 % × urgencia/100 + hasta 15 % por merma proyectada`, con tope de 65 %. Nunca se sugiere subir el precio.

## Por qué no Machine Learning (todavía)

Con pocas ventas por comercio, un modelo ML sobreajustaría y no sería explicable. El siguiente paso natural, con semanas de historial, es predecir demanda.

## Pruebas

`modelo_score.py` y `logica_ventas.py` son lógica pura (sin red ni Telegram), por lo que se pueden testear de forma aislada con `pytest`.